In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
import pandas as pd 
books=pd.read_csv("books_cleaned.csv")

In [4]:
books["tagged_description"]

0       9780002005883 A NOVEL THAT READERS and critics...
1       9780002261982 A new 'Christie for Christmas' -...
2       9780006178736 A memorable, mesmerizing heroine...
3       9780006280897 Lewis' work on the nature of lov...
4       9780006280934 "In The Problem of Pain, C.S. Le...
                              ...                        
5192    9788172235222 On A Train Journey Home To North...
5193    9788173031014 This book tells the tale of a ma...
5194    9788179921623 Wisdom to Create a Life of Passi...
5195    9788185300535 This collection of the timeless ...
5196    9789027712059 Since the three volume edition o...
Name: tagged_description, Length: 5197, dtype: object

In [5]:
#loader doesnt work with pandas dataframe
books["tagged_description"].to_csv(
    "tagged_description.txt",
    sep="\n",
    index=False,
    header=False,
    encoding="utf-8"
)

raw_documents = TextLoader("tagged_description.txt", encoding="utf-8").load()


In [6]:

text_splitter=CharacterTextSplitter(chunk_size=1000, chunk_overlap=0, separator="\n")
# If you set chunk_size=1000, and your descriptions are short (e.g., 200 characters each), the splitter might group 5 descriptions into a single chunk. In a recommendation or search context, this is usually bad because:

# It mixes data from different books.

# The vector embeddings would represent a "blur" of multiple items rather than a single specific book
documents=text_splitter.split_documents(raw_documents)

Created a chunk of size 1168, which is longer than the specified 1000
Created a chunk of size 1214, which is longer than the specified 1000
Created a chunk of size 1088, which is longer than the specified 1000
Created a chunk of size 1189, which is longer than the specified 1000
Created a chunk of size 1267, which is longer than the specified 1000
Created a chunk of size 2010, which is longer than the specified 1000
Created a chunk of size 1225, which is longer than the specified 1000
Created a chunk of size 1184, which is longer than the specified 1000
Created a chunk of size 1214, which is longer than the specified 1000
Created a chunk of size 1191, which is longer than the specified 1000
Created a chunk of size 1057, which is longer than the specified 1000
Created a chunk of size 1270, which is longer than the specified 1000
Created a chunk of size 1635, which is longer than the specified 1000
Created a chunk of size 1132, which is longer than the specified 1000
Created a chunk of s

In [7]:
documents[0]

Document(metadata={'source': 'tagged_description.txt'}, page_content='9780002005883 A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, Gi

In [8]:
db_books= Chroma.from_documents(
    documents, 
    embedding=OpenAIEmbeddings())

In [9]:
query="A book to teach children about nature"
docs=db_books.similarity_search(query,k=10)
docs

[Document(id='b1a1fd63-ad4a-46ff-bc2c-45dd9f316097', metadata={'source': 'tagged_description.txt'}, page_content="9780786808069 Children will discover the exciting world of their own backyard in this introduction to familiar animals from cats and dogs to bugs and frogs. The combination of photographs, illustrations, and fun facts make this an accessible and delightful learning experience.\n9780786808373 Introducing your baby to birds, cats, dogs, and babies through fine art, illsutration and photographs. These books are a rare opportunity to expose little ones to a range of images on a single subject, from simple child's drawings and abstract art to playful photos. A brief text accompanies each image, introducing baby to some basic -- and sometimes playful -- information on the subjects."),
 Document(id='40bf2f3b-4388-4675-b42f-63bcc2ef64df', metadata={'source': 'tagged_description.txt'}, page_content="9780786808380 Introduce your babies to birds, cats, dogs, and babies through fine ar

In [10]:
books[books["isbn13"]==int(docs[0].page_content.split()[0].strip())]

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
3747,9780786808069,0786808063,Baby Einstein: Neighborhood Animals,Marilyn Singer;Julie Aigner-Clark,Juvenile Fiction,http://books.google.com/books/content?id=X9a4P...,Children will discover the exciting world of t...,2001.0,3.89,16.0,180.0,Baby Einstein: Neighborhood Animals,9780786808069 Children will discover the excit...


In [11]:
def retreive_semantic_recommendations(
        query: str,
        top_k: int = 10,
    ) -> pd.DataFrame:
    recommendations = db_books.similarity_search(query, k=50)
    books_list = [
        int(recommendation.page_content.strip('"').split()[0])
        for recommendation in recommendations
    ]
    return books[books["isbn13"].isin(books_list)].head(top_k)

In [12]:
retreive_semantic_recommendations("A book to teach children about nature")

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
30,9780006646006,000664600X,Ocean Star Express,Mark Haddon;Peter Sutton,Juvenile Fiction,http://books.google.com/books/content?id=I2QZA...,Joe and his parents are enjoying a summer holi...,2002.0,3.50,32.0,1.0,Ocean Star Express,9780006646006 Joe and his parents are enjoying...
260,9780060852559,0060852550,"Animal, Vegetable, Miracle",Barbara Kingsolver;Camille Kingsolver;Steven L...,Biography & Autobiography,http://books.google.com/books/content?id=qLkEY...,Bestselling author Barbara Kingsolver returns ...,2007.0,4.04,370.0,86130.0,"Animal, Vegetable, Miracle: A Year of Food Life",9780060852559 Bestselling author Barbara Kings...
324,9780060959036,0060959037,Prodigal Summer,Barbara Kingsolver,Fiction,http://books.google.com/books/content?id=06IwG...,Barbara Kingsolver's fifth novel is a hymn to ...,2001.0,4.00,444.0,85440.0,Prodigal Summer: A Novel,9780060959036 Barbara Kingsolver's fifth novel...
399,9780062516374,006251637X,Rainforest Home Remedies,Rosita Arvigo;Nadine Epstein,Health & Fitness,http://books.google.com/books/content?id=DLnwO...,Rainforest Healing from Your Home and Garden F...,2001.0,4.12,221.0,65.0,Rainforest Home Remedies: The Maya Way To Heal...,9780062516374 Rainforest Healing from Your Hom...
413,9780064405850,0064405850,Strawberry Girl 60th Anniversary Edition,Lois Lenski,Juvenile Fiction,http://books.google.com/books/content?id=AQXM2...,"The land was theirs, but so were its hardships...",1995.0,3.86,208.0,10655.0,Strawberry Girl 60th Anniversary Edition,"9780064405850 The land was theirs, but so were..."
424,9780064410724,0064410722,Four Stupid Cupids,Gregory Maguire,Juvenile Fiction,http://books.google.com/books/content?id=471OU...,The students' scheme to find a love match for ...,2001.0,3.52,224.0,110.0,Four Stupid Cupids,9780064410724 The students' scheme to find a l...
442,9780067575208,006757520X,The Sense of Wonder,Rachel Carson,Nature,http://books.google.com/books/content?id=Zee5S...,"First published more than three decades ago, t...",1998.0,4.39,112.0,1160.0,The Sense of Wonder,9780067575208 First published more than three ...
879,9780152045739,0152045732,The Little Red Lighthouse and the Great Gray B...,Hildegarde Hoyt Swift;Lynd Ward,Juvenile Fiction,http://books.google.com/books/content?id=_bJOv...,A little lighthouse on the Hudson River regain...,2003.0,4.26,64.0,2216.0,The Little Red Lighthouse and the Great Gray B...,9780152045739 A little lighthouse on the Hudso...
1077,9780240806082,0240806085,Directing the Documentary,Michael Rabiger,Performing Arts,http://books.google.com/books/content?id=uoKli...,Michael Rabiger guides the reader through the ...,2004.0,4.23,648.0,173.0,Directing the Documentary,9780240806082 Michael Rabiger guides the reade...
1220,9780312315733,0312315732,Little Children,Tom Perrotta,Fiction,http://books.google.com/books/content?id=j73yQ...,"A group of young suburban parents, including a...",2004.0,3.62,355.0,25811.0,Little Children: A Novel,9780312315733 A group of young suburban parent...


In [13]:
category_mapping = {'Fiction' : "Fiction",
'Juvenile Fiction': "Children's Fiction",
'Biography & Autobiography': "Nonfiction",
'History': "Nonfiction",
'Literary Criticism': "Nonfiction",
'Philosophy': "Nonfiction",
'Religion': "Nonfiction",
'Comics & Graphic Novels': "Fiction",
'Drama': "Fiction",
'Juvenile Nonfiction': "Children's Nonfiction",
'Science': "Nonfiction",
'Poetry': "Fiction"}

In [ ]:
#books["categories"].value_counts().reset_index()

,categories,count
0,Fiction,2111
1,Juvenile Fiction,390
2,Biography & Autobiography,311
3,History,207
4,Literary Criticism,124
...,...,...
474,Human-animal relationships,1
475,Imperialism,1
476,Aged women,1
477,Humorous stories,1


In [14]:
books["simple_categories"]=books["categories"].map(category_mapping)
books[~(books["simple_categories"].isna())]

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description,simple_categories
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0,Gilead,9780002005883 A NOVEL THAT READERS and critics...,Fiction
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0,Rage of angels,"9780006178736 A memorable, mesmerizing heroine...",Fiction
8,9780006482079,0006482074,Warhost of Vastmark,Janny Wurts,Fiction,http://books.google.com/books/content?id=uOL0f...,"Tricked once more by his wily half-brother, Ly...",1995.0,4.03,522.0,2966.0,Warhost of Vastmark,9780006482079 Tricked once more by his wily ha...,Fiction
30,9780006646006,000664600X,Ocean Star Express,Mark Haddon;Peter Sutton,Juvenile Fiction,http://books.google.com/books/content?id=I2QZA...,Joe and his parents are enjoying a summer holi...,2002.0,3.50,32.0,1.0,Ocean Star Express,9780006646006 Joe and his parents are enjoying...,Children's Fiction
46,9780007121014,0007121016,Taken at the Flood,Agatha Christie,Fiction,http://books.google.com/books/content?id=3gWlx...,A Few Weeks After Marrying An Attractive Young...,2002.0,3.71,352.0,8852.0,Taken at the Flood,9780007121014 A Few Weeks After Marrying An At...,Fiction
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5178,9781933648279,1933648279,Night Has a Thousand Eyes,Cornell Woolrich,Fiction,http://books.google.com/books/content?id=3Gk6s...,"""Cornell Woolrich's novels define the essence ...",2007.0,3.77,344.0,680.0,Night Has a Thousand Eyes,"9781933648279 ""Cornell Woolrich's novels defin...",Fiction
5188,9784770028969,4770028962,Coin Locker Babies,村上龍,Fiction,http://books.google.com/books/content?id=87DJw...,Rescued from the lockers in which they were le...,2002.0,3.75,393.0,5560.0,Coin Locker Babies,9784770028969 Rescued from the lockers in whic...,Fiction
5189,9788122200850,8122200850,"Cry, the Peacock",Anita Desai,Fiction,http://books.google.com/books/content?id=_QKwV...,This book is the story of a young girl obsesse...,1980.0,3.22,218.0,134.0,"Cry, the Peacock",9788122200850 This book is the story of a youn...,Fiction
5195,9788185300535,8185300534,I Am that,Sri Nisargadatta Maharaj;Sudhakar S. Dikshit,Philosophy,http://books.google.com/books/content?id=Fv_JP...,This collection of the timeless teachings of o...,1999.0,4.51,531.0,104.0,I Am that: Talks with Sri Nisargadatta Maharaj,9788185300535 This collection of the timeless ...,Nonfiction


In [65]:
#zero-shot classificafication: 
from transformers import pipeline
pipe = pipeline("zero-shot-classification", 
                model="facebook/bart-large-mnli")

Device set to use cpu


In [72]:
# Check available pipeline variables
print("Available variables with 'pipe' in name:")
for name in dir():
    if 'pipe' in name.lower():
        print(f"  - {name}: {type(eval(name))}")

Available variables with 'pipe' in name:
  - pipe: <class 'transformers.pipelines.zero_shot_classification.ZeroShotClassificationPipeline'>
  - pipe3: <class 'transformers.pipelines.zero_shot_classification.ZeroShotClassificationPipeline'>
  - pipeline: <class 'function'>


In [73]:
# Test generate_predictions function directly
try:
    result = generate_predictions("A story about love and loss", ["Fiction", "Nonfiction"])
    print(f"Function works! Result: {result}")
except Exception as e:
    print(f"Error: {e}")

Function works! Result: Fiction


In [26]:
book_categories= ["Fiction", "NonFiction"]
sequence= books.loc[books["simple_categories"]== "Fiction", "description"].reset_index(drop=True)[0]
pipe(sequence, book_categories)

{'sequence': 'A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, Gilead is a song of celebration and acceptance of the best and the worst

In [27]:
def generate_predictions(sequence, categories):
    predictions = pipe(sequence, categories)
    max_index = np.argmax(predictions["scores"])
    max_label = predictions["labels"][max_index]
    return max_label

In [37]:
from tqdm import tqdm

actual_cats = []
predicted_cats = []

for i in tqdm(range(0, 300)):
    sequence = books.loc[books["simple_categories"] == "Fiction", "description"].reset_index(drop=True)[i]
    predicted_cats += [generate_predictions(sequence, book_categories)]
    actual_cats += ["Fiction"]

100%|██████████| 300/300 [12:58<00:00,  2.60s/it]


In [29]:
for i in tqdm(range(0, 300)):
    sequence = books.loc[books["simple_categories"] == "Nonfiction", "description"].reset_index(drop=True)[i]
    predicted_cats += [generate_predictions(sequence, book_categories)]
    actual_cats += ["Nonfiction"]

100%|██████████| 300/300 [11:19<00:00,  2.26s/it]


In [41]:
predictions_dataframe=pd.DataFrame({"actual_categories":actual_cats, "predicted_categories":predicted_cats})
predictions_dataframe

,actual_categories,predicted_categories
0,Fiction,Fiction
1,Fiction,NonFiction
2,Fiction,NonFiction
3,Fiction,NonFiction
4,Fiction,NonFiction
...,...,...
295,Fiction,NonFiction
296,Fiction,NonFiction
297,Fiction,NonFiction
298,Fiction,NonFiction


In [42]:
predictions_dataframe["correct_predictions"]=(np.where(predictions_dataframe["actual_categories"]==predictions_dataframe["predicted_categories"],1,0))

In [43]:
predictions_dataframe["correct_predictions"].sum()/len(predictions_dataframe)

np.float64(0.43)

In [47]:
# Test with a better model specifically designed for zero-shot classification
from transformers import pipeline
pipe2 = pipeline("zero-shot-classification", 
                model="typeform/distilbert-base-uncased-mnli")

Device set to use cpu


In [48]:
# Test the new model on a few samples
book_categories = ["Fiction", "Nonfiction"]
for i in range(3):
    seq = books.loc[books["simple_categories"] == "Fiction", "description"].reset_index(drop=True)[i]
    result = pipe2(seq, book_categories)
    print(f"Sample {i}: Predicted={result['labels'][0]}, Score={result['scores'][0]:.3f}")

Sample 0: Predicted=Fiction, Score=0.627
Sample 1: Predicted=Fiction, Score=0.575
Sample 2: Predicted=Nonfiction, Score=0.510


In [49]:
# Run predictions with the new model on all 600 samples
actual_cats2 = []
predicted_cats2 = []

for i in tqdm(range(0, 300)):
    sequence = books.loc[books["simple_categories"] == "Fiction", "description"].reset_index(drop=True)[i]
    predictions = pipe2(sequence, book_categories)
    max_index = np.argmax(predictions["scores"])
    predicted_cats2 += [predictions["labels"][max_index]]
    actual_cats2 += ["Fiction"]

for i in tqdm(range(0, 300)):
    sequence = books.loc[books["simple_categories"] == "Nonfiction", "description"].reset_index(drop=True)[i]
    predictions = pipe2(sequence, book_categories)
    max_index = np.argmax(predictions["scores"])
    predicted_cats2 += [predictions["labels"][max_index]]
    actual_cats2 += ["Nonfiction"]

100%|██████████| 300/300 [01:44<00:00,  2.88it/s]


In [50]:
# Calculate accuracy with new model
predictions_dataframe2 = pd.DataFrame({"actual_categories":actual_cats2, "predicted_categories":predicted_cats2})
predictions_dataframe2["correct_predictions"] = (np.where(predictions_dataframe2["actual_categories"]==predictions_dataframe2["predicted_categories"],1,0))
accuracy2 = predictions_dataframe2["correct_predictions"].sum()/len(predictions_dataframe2)
print(f"New Model Accuracy: {accuracy2:.3f}")

New Model Accuracy: 0.657


In [51]:
# Try another model - roberta-large for potentially better results
pipe3 = pipeline("zero-shot-classification", 
                model="roberta-large-mnli")

config.json:   0%|          | 0.00/688 [00:00<?, ?B/s]

d:\SemanticBookRecommender\venv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP\.cache\huggingface\hub\models--roberta-large-mnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/1.43G [00:00<?, ?B/s]

Some weights of the model checkpoint at roberta-large-mnli were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


In [52]:
# Test roberta model on a few samples
for i in range(3):
    seq = books.loc[books["simple_categories"] == "Fiction", "description"].reset_index(drop=True)[i]
    result = pipe3(seq, book_categories)
    print(f"Sample {i}: Predicted={result['labels'][0]}, Score={result['scores'][0]:.3f}")

Sample 0: Predicted=Fiction, Score=0.630
Sample 1: Predicted=Fiction, Score=0.670
Sample 2: Predicted=Fiction, Score=0.630


In [53]:
# Run predictions with roberta model on all 600 samples
actual_cats3 = []
predicted_cats3 = []

for i in tqdm(range(0, 300)):
    sequence = books.loc[books["simple_categories"] == "Fiction", "description"].reset_index(drop=True)[i]
    predictions = pipe3(sequence, book_categories)
    max_index = np.argmax(predictions["scores"])
    predicted_cats3 += [predictions["labels"][max_index]]
    actual_cats3 += ["Fiction"]

for i in tqdm(range(0, 300)):
    sequence = books.loc[books["simple_categories"] == "Nonfiction", "description"].reset_index(drop=True)[i]
    predictions = pipe3(sequence, book_categories)
    max_index = np.argmax(predictions["scores"])
    predicted_cats3 += [predictions["labels"][max_index]]
    actual_cats3 += ["Nonfiction"]

100%|██████████| 300/300 [10:45<00:00,  2.15s/it]


In [54]:
# Calculate accuracy with roberta model
predictions_dataframe3 = pd.DataFrame({"actual_categories":actual_cats3, "predicted_categories":predicted_cats3})
predictions_dataframe3["correct_predictions"] = (np.where(predictions_dataframe3["actual_categories"]==predictions_dataframe3["predicted_categories"],1,0))
accuracy3 = predictions_dataframe3["correct_predictions"].sum()/len(predictions_dataframe3)
print(f"RoBERTa Accuracy: {accuracy3:.3f}")

RoBERTa Accuracy: 0.830


In [59]:
# Remove old models to save space
# Delete all pipeline variables except the best one (pipe3)
try:
    del pipe
    print("Deleted: pipe")
except:
    pass

try:
    del pipe2
    print("Deleted: pipe2")
except:
    pass

# Force garbage collection
import gc
gc.collect()
print("Memory cleaned!")

Memory cleaned!


In [74]:
#finding the isbns having missing labels and predicting their labels
isbns = []
predicted_cats = []

missing_cats = books.loc[books["simple_categories"].isna(), ["isbn13", "description"]].reset_index(drop=True)

for i in tqdm(range(0, len(missing_cats))):
    sequence = missing_cats["description"][i]
    predicted_cats += [generate_predictions(sequence, book_categories)]
    isbns += [missing_cats["isbn13"][i]]

100%|██████████| 1454/1454 [50:12<00:00,  2.07s/it] 


missing_predicted_dataframe=pd.DataFrame({"isbn13"})

In [76]:
missing_predicted_dataframe=pd.DataFrame({"isbn13":isbns, "predicted_categories":predicted_cats})
missing_predicted_dataframe

,isbn13,predicted_categories
0,9780002261982,Fiction
1,9780006280897,Nonfiction
2,9780006280934,Nonfiction
3,9780006380832,Nonfiction
4,9780006470229,Fiction
...,...,...
1449,9788125026600,Nonfiction
1450,9788171565641,Fiction
1451,9788172235222,Fiction
1452,9788173031014,Nonfiction


In [81]:

books=pd.merge(books,missing_predicted_dataframe, on="isbn13", how="left")
#when the orignal simple coloumns categories is misisng , please use predicted categories.
books["simple_categores"]=np.where(books["simple_categories"].isna(),books["predicted_cats"], books["simple_categores"])
books

MergeError: Passing 'suffixes' which cause duplicate columns {'predicted_categories_x'} is not allowed.

In [82]:
# Check the structure of both dataframes
print("books columns:", books.columns.tolist())
print("\nmissing_predicted_dataframe columns:", missing_predicted_dataframe.columns.tolist())

books columns: ['isbn13', 'isbn10', 'title', 'authors', 'categories', 'thumbnail', 'description', 'published_year', 'average_rating', 'num_pages', 'ratings_count', 'title_and_subtitle', 'tagged_description', 'simple_categories', 'predicted_categories_x', 'predicted_categories_y', 'predicted_categories']

missing_predicted_dataframe columns: ['isbn13', 'predicted_categories']


In [ ]:
# Drop duplicate columns and do clean merge
# First, reload books from CSV to get clean data
books = pd.read_csv("books_cleaned.csv")
books["simple_categories"] = books["categories"].map(category_mapping)

# Now merge with predicted categories
books = pd.merge(books, missing_predicted_dataframe, on="isbn13", how="left")

# Fill missing categories with predictions
books["simple_categories"] = np.where(
    books["simple_categories"].isna(), 
    books["predicted_categories"], 
    books["simple_categories"]
)

# Drop the extra column
books = books.drop(columns=["predicted_categories"])

print("Ctaegories of the books")
books["simple_categories"].value_counts()


Success! Books with categories:


simple_categories
Fiction                  2808
Nonfiction               1942
Children's Fiction        390
Children's Nonfiction      57
Name: count, dtype: int64

In [ ]:
books.to_csv("books_with_categories.csv", index=False)

<bound method IndexOpsMixin.value_counts of 0       False
1       False
2       False
3       False
4       False
        ...  
5192    False
5193    False
5194    False
5195    False
5196    False
Name: simple_categories, Length: 5197, dtype: bool>

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description,simple_categories,predicted_categories_x,predicted_categories_y,predicted_categories,simple_categores
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0,Gilead,9780002005883 A NOVEL THAT READERS and critics...,Fiction,NaN,NaN,NaN,Fiction
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0,Spider's Web: A Novel,9780002261982 A new 'Christie for Christmas' -...,Fiction,Fiction,Fiction,Fiction,Fiction
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0,Rage of angels,"9780006178736 A memorable, mesmerizing heroine...",Fiction,NaN,NaN,NaN,Fiction
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0,The Four Loves,9780006280897 Lewis' work on the nature of lov...,Nonfiction,Nonfiction,Nonfiction,Nonfiction,Nonfiction
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,37569.0,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. Le...",Nonfiction,Nonfiction,Nonfiction,Nonfiction,Nonfiction
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5192,9788172235222,8172235224,Mistaken Identity,Nayantara Sahgal,Indic fiction (English),http://books.google.com/books/content?id=q-tKP...,On A Train Journey Home To North India After L...,2003.0,2.93,324.0,0.0,Mistaken Identity,9788172235222 On A Train Journey Home To North...,Fiction,Fiction,Fiction,Fiction,Fiction
5193,9788173031014,8173031010,Journey to the East,Hermann Hesse,Adventure stories,http://books.google.com/books/content?id=rq6JP...,This book tells the tale of a man who goes on ...,2002.0,3.70,175.0,24.0,Journey to the East,9788173031014 This book tells the tale of a ma...,Nonfiction,Nonfiction,Nonfiction,Nonfiction,Nonfiction
5194,9788179921623,817992162X,The Monk Who Sold His Ferrari: A Fable About F...,Robin Sharma,Health & Fitness,http://books.google.com/books/content?id=c_7mf...,"Wisdom to Create a Life of Passion, Purpose, a...",2003.0,3.82,198.0,1568.0,The Monk Who Sold His Ferrari: A Fable About F...,9788179921623 Wisdom to Create a Life of Passi...,Fiction,Fiction,Fiction,Fiction,Fiction
5195,9788185300535,8185300534,I Am that,Sri Nisargadatta Maharaj;Sudhakar S. Dikshit,Philosophy,http://books.google.com/books/content?id=Fv_JP...,This collection of the timeless teachings of o...,1999.0,4.51,531.0,104.0,I Am that: Talks with Sri Nisargadatta Maharaj,9788185300535 This collection of the timeless ...,Nonfiction,NaN,NaN,NaN,Nonfiction
